In [3]:
import sys, json, random
from pathlib import Path
from collections import defaultdict
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists():   # walk up until we find the repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.prompts import build_messages, format_answer
from src.evaluate import score_prediction
from src.abstention import build_abstention_records

SPLITS = ROOT / "notebooks/data/splits"
OUT = ROOT / "notebooks/data/dpo"
OUT.mkdir(parents=True, exist_ok=True)

pool = [json.loads(l) for l in open(SPLITS / "dpo_pool.jsonl", encoding="utf-8")]
pool_by_id = {r["id"]: r for r in pool}
random.seed(3407)
print(len(pool), "records")
print(pool[0]["query"], "\n->", format_answer(pool[0]["answers"]))

5863 records
Is 'https://www.apple.com' available in the Wayback Machine on September 9, 2015? 
-> [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]


In [4]:
from openai import OpenAI   # pip install openai if missing
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
MODEL = "sft-toolcaller"
print(client.models.list().data[0].id)   # check vLLM is up

def sample(record, n=4, temperature=0.8):
    resp = client.chat.completions.create(
        model=MODEL, messages=build_messages(record),
        n=n, temperature=temperature, top_p=0.95, max_tokens=512,
    )
    return [c.message.content for c in resp.choices]

rec = pool[0]
for t in sample(rec):
    print(t)

sft-toolcaller
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909", "callback": null}}]
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
[{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]


In [5]:
for t in sample(rec):
    s = score_prediction(t, rec)
    print(s["exact_match"], s["failure_type"], "|", t[:120])

1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]
1 correct | [{"name": "availability", "arguments": {"url": "https://www.apple.com", "timestamp": "20150909"}}]


In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def sample_many(records, out_path, n=4, workers=16):
    out_path = Path(out_path)
    done = set()
    if out_path.exists():
        done = {json.loads(l)["id"] for l in open(out_path, encoding="utf-8")}
    todo = [r for r in records if r["id"] not in done]
    print(f"{len(done)} done, {len(todo)} to go")

    def work(r):
        try:
            return {"id": r["id"], "responses": sample(r, n=n)}
        except Exception as e:            # e.g. prompt longer than max-model-len
            return {"id": r["id"], "responses": [], "error": str(e)[:200]}

    with open(out_path, "a", encoding="utf-8") as f, ThreadPoolExecutor(workers) as ex:
        futures = [ex.submit(work, r) for r in todo]
        for fut in tqdm(as_completed(futures), total=len(futures)):
            f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n")
            f.flush()

# smoke test first, then the full run (it skips what's already done)
#sample_many(pool[:50], OUT / "samples_pool.jsonl")
sample_many(pool, OUT / "samples_pool.jsonl")

50 done, 5813 to go


  0%|          | 0/5813 [00:00<?, ?it/s]

In [8]:
from collections import Counter

samples = [json.loads(l) for l in open(OUT / "samples_pool.jsonl", encoding="utf-8")]
rows = []
for s in samples:
    rec = pool_by_id[s["id"]]
    for t in s["responses"]:
        sc = score_prediction(t, rec)
        rows.append({"id": s["id"], "text": t.strip(), "exact": sc["exact_match"], "failure_type": sc["failure_type"]})
df = pd.DataFrame(rows)

print("sample accuracy:", round(df.exact.mean() * 100, 1), "%")
print(df[df.exact == 0].failure_type.value_counts())

def make_pair(rec, chosen, rejected, kind):
    return {"id": rec["id"], "kind": kind, "prompt": build_messages(rec),
            "chosen":   [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}]}

mistake_pairs = []
for rid, g in df.groupby("id"):
    wrong, right = g[g.exact == 0], g[g.exact == 1]
    if wrong.empty:
        continue
    rec = pool_by_id[rid]
    chosen = right.text.iloc[0] if len(right) else format_answer(rec["answers"])
    bad_text = wrong.text.value_counts().index[0]          # (1) the model's most common mistake
    if bad_text == chosen:
        continue
    p = make_pair(rec, chosen, bad_text, "mistake")
    p["failure_type"] = wrong[wrong.text == bad_text].failure_type.iloc[0]
    p["all_wrong_same"] = len(right) == 0 and wrong.text.nunique() == 1
    mistake_pairs.append(p)

# (3) eyeball possible label noise
sus = [p for p in mistake_pairs if p["all_wrong_same"]]
print(f"\n{len(sus)} pairs where all 4 samples gave the SAME wrong answer. Check a few:")
for p in random.sample(sus, min(3, len(sus))):
    print("\nQ:   ", pool_by_id[p["id"]]["query"])
    print("GOLD:", p["chosen"][0]["content"])
    print("MODEL:", p["rejected"][0]["content"])

# (2) cap per failure type
CAP = 1000
by_type = defaultdict(list)
for p in mistake_pairs:
    by_type[p["failure_type"]].append(p)
mistake_pairs = []
for ft, ps in by_type.items():
    random.shuffle(ps)
    mistake_pairs += ps[:CAP]
print("\n", len(mistake_pairs), "mistake pairs", {k: min(len(v), CAP) for k, v in by_type.items()})

sample accuracy: 79.3 %
failure_type
wrong_value             4240
wrong_call_count         336
wrong_tool               183
parse_fail                44
unknown_arg               34
missing_required_arg      14
invented_tool              3
Name: count, dtype: int64

228 pairs where all 4 samples gave the SAME wrong answer. Check a few:

Q:    I need the Twitter details for '@LadyGaga' and TikTok videos on 'music' sorted by relevance.
GOLD: [{"name": "user_by_screen_name", "arguments": {"username": "LadyGaga"}}, {"name": "get_related_video_with_keywords", "arguments": {"keywords": "music", "count": "10"}}]
MODEL: [{"name": "user_by_screen_name", "arguments": {"username": "LadyGaga"}}, {"name": "get_related_video_with_keywords", "arguments": {"keywords": "music", "sort_type": 0}}]

Q:    Calculate the weighted average grade for a student who scored 75, 82, 91, and 88 in four exams, with respective weights of 0.2, 0.3, 0.25, and 0.25. Also, generate a random string of length 10 that inclu

In [9]:
abstain_recs = build_abstention_records(pool, n=1000, seed=3407)
abstain_by_id = {r["id"]: r for r in abstain_recs}
sample_many(abstain_recs, OUT / "samples_abstain.jsonl")      # prints "1000 done, 0 to go"

abstain_pairs = []
for l in open(OUT / "samples_abstain.jsonl", encoding="utf-8"):
    s = json.loads(l)
    rec = abstain_by_id[s["id"]]
    invented = [t.strip() for t in s["responses"]
                if score_prediction(t, rec)["failure_type"] == "called_tool_when_none_needed"]
    if invented:
        top = Counter(invented).most_common(1)[0][0]           # (1) most common invented call
        p = make_pair(rec, "[]", top, "abstain")
        p["source_id"], p["neg_kind"] = rec["source_id"], rec["neg_kind"]
        abstain_pairs.append(p)
print(len(abstain_pairs), "abstain pairs available", Counter(p["neg_kind"] for p in abstain_pairs))

1000 done, 0 to go


0it [00:00, ?it/s]

999 abstain pairs available Counter({'removed': 500, 'swapped': 499})


In [11]:
# keep abstain at ~18% of the total, half hard (removed) / half easy (swapped)   (4)
n_abs = int(0.18 * len(mistake_pairs) / (1 - 2 * 0.18))
removed = [p for p in abstain_pairs if p["neg_kind"] == "removed"]
swapped = [p for p in abstain_pairs if p["neg_kind"] == "swapped"]
abstain_pairs = removed[: n_abs // 2]
abstain_pairs += swapped[: n_abs - len(abstain_pairs)]

balance_pairs = []
for p in abstain_pairs:
    rec = pool_by_id[p["source_id"]]
    balance_pairs.append(make_pair(rec, format_answer(rec["answers"]), "[]", "balance"))
print(len(abstain_pairs), "abstain", Counter(p["neg_kind"] for p in abstain_pairs), "|", len(balance_pairs), "balance")

336 abstain Counter({'removed': 168, 'swapped': 168}) | 336 balance


In [12]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(ROOT / "notebooks/outputs/sft_qwen2.5-0.5b/merged")

all_pairs = mistake_pairs + abstain_pairs + balance_pairs
def n_tokens(p, side):
    return len(tok.apply_chat_template(p["prompt"] + p[side], tokenize=True))

lens = pd.Series([max(n_tokens(p, "chosen"), n_tokens(p, "rejected")) for p in all_pairs])
print(lens.describe(percentiles=[.5, .9, .95, .99]).round(0))

MAX_LEN = 2048                                                  # (5)
all_pairs = [p for p, L in zip(all_pairs, lens) if L <= MAX_LEN]
print(len(all_pairs), "pairs kept after length filter")

count    1869.0
mean        2.0
std         0.0
min         2.0
50%         2.0
90%         2.0
95%         2.0
99%         2.0
max         2.0
dtype: float64
1869 pairs kept after length filter


In [13]:
group = lambda p: p.get("source_id", p["id"])
groups = sorted({group(p) for p in all_pairs})
random.shuffle(groups)
val_groups = set(groups[: int(0.05 * len(groups))])

train = [p for p in all_pairs if group(p) not in val_groups]
val   = [p for p in all_pairs if group(p) in val_groups]
random.shuffle(train)

for name, rows in [("dpo_train", train), ("dpo_val", val)]:
    with open(OUT / f"{name}.jsonl", "w", encoding="utf-8") as f:
        for p in rows:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(pd.Series([p["kind"] for p in all_pairs]).value_counts())
print(len(train), "train /", len(val), "val")
for kind in ["mistake", "abstain", "balance"]:
    ex = random.choice([p for p in train if p["kind"] == kind])
    print(f"\n[{kind}]\nCHOSEN:   {ex['chosen'][0]['content']}\nREJECTED: {ex['rejected'][0]['content']}")

mistake    1197
abstain     336
balance     336
Name: count, dtype: int64
1772 train / 97 val

[mistake]
CHOSEN:   [{"name": "image_search", "arguments": {"q": "Pyramids of Giza", "num": 10}}, {"name": "web_search", "arguments": {"query": "Egypt", "limit": 10}}]
REJECTED: [{"name": "image_search", "arguments": {"q": "Pyramids of Giza"}}, {"name": "web_search", "arguments": {"query": "news articles about Egypt"}}]

[abstain]
CHOSEN:   []
REJECTED: [{"name": "get_random_chess_puzzles", "arguments": {"number_of_puzzles": "5"}}, {"name": "get_track_of_the_day", "arguments": {"start_date": "2022-02-01", "end_date": "2022-02-31"}}]

[balance]
CHOSEN:   [{"name": "v1_iplookup", "arguments": {"address": "192.168.1.100"}}, {"name": "v1_iplookup", "arguments": {"address": "172.16.0.1"}}]
REJECTED: []
